In [1]:
# PHASE 10 — STATISTICAL ANALYSIS
# H1: Does late delivery significantly affect customer ratings?

import pandas as pd
import numpy as np
from scipy.stats import ttest_ind
from math import sqrt

# --------------------------------------------------
# 1. Load data
# --------------------------------------------------

orders = pd.read_csv("../data/olist_orders_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")


# --------------------------------------------------
# 2. Convert dates
# --------------------------------------------------

orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"],
    errors="coerce"
)

orders["order_estimated_delivery_date"] = pd.to_datetime(
    orders["order_estimated_delivery_date"],
    errors="coerce"
)


# --------------------------------------------------
# 3. Calculate delivery delay
# --------------------------------------------------

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / (60 * 60 * 24)


# --------------------------------------------------
# 4. Create delivery groups
# --------------------------------------------------

orders["delivery_status"] = np.where(
    orders["delivery_delay_days"] > 0,
    "Late",
    "On-time/Early"
)


# --------------------------------------------------
# 5. Merge with reviews
# --------------------------------------------------

h1_data = orders[
    ["order_id", "delivery_delay_days", "delivery_status"]
].merge(
    reviews[
        ["order_id", "review_score"]
    ],
    on="order_id",
    how="inner"
)


# --------------------------------------------------
# 6. Remove missing values
# --------------------------------------------------

h1_data = h1_data.dropna(
    subset=["delivery_delay_days", "review_score"]
)


# --------------------------------------------------
# 7. Separate groups
# --------------------------------------------------

late_scores = h1_data.loc[
    h1_data["delivery_status"] == "Late",
    "review_score"
]

on_time_scores = h1_data.loc[
    h1_data["delivery_status"] == "On-time/Early",
    "review_score"
]


# --------------------------------------------------
# 8. Sample sizes and means
# --------------------------------------------------

n_late = len(late_scores)
n_on_time = len(on_time_scores)

mean_late = late_scores.mean()
mean_on_time = on_time_scores.mean()

difference = mean_on_time - mean_late

print("Sample sizes:")
print("Late:", n_late)
print("On-time/Early:", n_on_time)

print("\nAverage review scores:")
print("Late:", round(mean_late, 2))
print("On-time/Early:", round(mean_on_time, 2))

print("\nDifference in average review score:")
print(round(difference, 2))


# --------------------------------------------------
# 9. Welch's t-test
# --------------------------------------------------

t_statistic, p_value = ttest_ind(
    late_scores,
    on_time_scores,
    equal_var=False
)

print("\nWelch's t-test:")
print("T-statistic:", round(t_statistic, 4))
print("P-value:", p_value)


# --------------------------------------------------
# 10. Cohen's d effect size
# --------------------------------------------------

sd_late = late_scores.std()
sd_on_time = on_time_scores.std()

pooled_sd = sqrt(
    (
        (n_late - 1) * sd_late**2
        +
        (n_on_time - 1) * sd_on_time**2
    )
    /
    (n_late + n_on_time - 2)
)

cohens_d = difference / pooled_sd

print("\nEffect size:")
print("Cohen's d:", round(cohens_d, 4))


# --------------------------------------------------
# 11. 95% Confidence Interval
# --------------------------------------------------

from scipy.stats import t

se = sqrt(
    (sd_late**2 / n_late)
    +
    (sd_on_time**2 / n_on_time)
)

# Welch-Satterthwaite degrees of freedom

df = (
    (
        sd_late**2 / n_late
        +
        sd_on_time**2 / n_on_time
    ) ** 2
    /
    (
        (sd_late**2 / n_late) ** 2 / (n_late - 1)
        +
        (sd_on_time**2 / n_on_time) ** 2 / (n_on_time - 1)
    )
)

critical_value = t.ppf(
    0.975,
    df
)

margin_of_error = critical_value * se

ci_lower = difference - margin_of_error
ci_upper = difference + margin_of_error

print("\n95% Confidence Interval:")
print(
    "Lower:", round(ci_lower, 4)
)

print(
    "Upper:", round(ci_upper, 4)
)


# --------------------------------------------------
# 12. Business interpretation
# --------------------------------------------------

print("\nBUSINESS INTERPRETATION:")

if p_value < 0.05:
    print(
        "Late delivery is significantly associated "
        "with lower customer ratings."
    )
else:
    print(
        "There is not enough statistical evidence "
        "to conclude that late delivery is associated "
        "with lower customer ratings."
    )

Sample sizes:
Late: 7701
On-time/Early: 88658

Average review scores:
Late: 2.57
On-time/Early: 4.29

Difference in average review score:
1.73

Welch's t-test:
T-statistic: -89.5508
P-value: 0.0

Effect size:
Cohen's d: 1.4431

95% Confidence Interval:
Lower: 1.6892
Upper: 1.7648

BUSINESS INTERPRETATION:
Late delivery is significantly associated with lower customer ratings.


In [2]:
# PHASE 10 — STATISTICAL ANALYSIS
# H4: Does freight cost significantly affect customer ratings?

import pandas as pd
import numpy as np
from scipy.stats import spearmanr

# Load data
order_items = pd.read_csv("../data/olist_order_items_dataset.csv")
reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

# Calculate total price and total freight per order
order_financials = (
    order_items
    .groupby("order_id")
    .agg(
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
    .reset_index()
)

# Calculate freight-cost ratio
order_financials["freight_ratio"] = np.where(
    order_financials["total_price"] > 0,
    (order_financials["total_freight"] /
     order_financials["total_price"]) * 100,
    np.nan
)

# Merge with reviews
h4_data = order_financials.merge(
    reviews[["order_id", "review_score"]],
    on="order_id",
    how="inner"
)

# Remove missing values
h4_data = h4_data.dropna(
    subset=["freight_ratio", "review_score"]
)

print("Sample size:", len(h4_data))

print("\nAverage values:")
print("Average freight ratio:",
      round(h4_data["freight_ratio"].mean(), 2), "%")

print("Average review score:",
      round(h4_data["review_score"].mean(), 2))

# Spearman correlation
rho, p_value = spearmanr(
    h4_data["freight_ratio"],
    h4_data["review_score"]
)

print("\nSpearman correlation:")
print("Correlation (rho):", round(rho, 4))
print("P-value:", p_value)

# 95% Confidence Interval using Fisher transformation
n = len(h4_data)

z = np.arctanh(rho)
se = 1 / np.sqrt(n - 3)

z_lower = z - 1.96 * se
z_upper = z + 1.96 * se

ci_lower = np.tanh(z_lower)
ci_upper = np.tanh(z_upper)

print("\n95% Confidence Interval:")
print("Lower:", round(ci_lower, 4))
print("Upper:", round(ci_upper, 4))

# Verdict
print("\nH4 VERDICT:")

if p_value < 0.05:
    print("CONFIRMED")
else:
    print("INCONCLUSIVE")

print("\nBUSINESS INTERPRETATION:")

if p_value < 0.05:
    print(
        "Higher freight-cost ratios are statistically associated "
        "with lower customer ratings."
    )
else:
    print(
        "There is not enough statistical evidence to conclude "
        "that freight-cost ratio is associated with customer ratings."
    )

Sample size: 98465

Average values:
Average freight ratio: 30.86 %
Average review score: 4.1

Spearman correlation:
Correlation (rho): -0.026
P-value: 3.3549334970002794e-16

95% Confidence Interval:
Lower: -0.0322
Upper: -0.0198

H4 VERDICT:
CONFIRMED

BUSINESS INTERPRETATION:
Higher freight-cost ratios are statistically associated with lower customer ratings.
